# Environment test

Run each cell to verify your Kaggle competition environment is set up correctly. All imports should succeed, and the version assertions confirm the environment matches the Kaggle notebook runtime.

In [ ]:
import sys
print(f"Python {sys.version}")
assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version}"
print("Python version: OK")

## Core data science packages

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import scipy

print(f"numpy  {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"polars {pl.__version__}")
print(f"scipy  {scipy.__version__}")

# Pinned to match the Kaggle notebook environment
assert np.__version__ == "2.0.2", np.__version__
assert pd.__version__ == "2.3.3", pd.__version__
print("Versions match Kaggle: OK")

In [ ]:
import sklearn
import xgboost as xgb
import lightgbm as lgb
import catboost
import statsmodels

print(f"scikit-learn {sklearn.__version__}")
print(f"xgboost      {xgb.__version__}")
print(f"lightgbm     {lgb.__version__}")
print(f"catboost     {catboost.__version__}")
print(f"statsmodels  {statsmodels.__version__}")

# Pinned to match the Kaggle notebook environment
assert sklearn.__version__ == "1.6.1", sklearn.__version__
assert xgb.__version__ == "3.2.0", xgb.__version__
print("Versions match Kaggle: OK")

## Visualization packages

In [ ]:
import matplotlib
import seaborn as sns
import plotly

print(f"matplotlib {matplotlib.__version__}")
print(f"seaborn    {sns.__version__}")
print(f"plotly     {plotly.__version__}")

## Hyperparameter optimization

In [ ]:
import optuna

print(f"optuna {optuna.__version__}")

## Deep learning frameworks

In [ ]:
import torch
import tensorflow as tf
import keras

print(f"torch      {torch.__version__}")
print(f"tensorflow {tf.__version__}")
print(f"keras      {keras.__version__}")

# Pinned to match the Kaggle notebook environment
assert torch.__version__.startswith("2.10.0"), torch.__version__
assert tf.__version__ == "2.20.0", tf.__version__
print("Versions match Kaggle: OK")

## HuggingFace and Kaggle tooling

In [ ]:
import transformers
import kagglehub

print(f"transformers {transformers.__version__}")
print(f"kagglehub    {kagglehub.__version__}")

# Kaggle CLI (requires API credentials for actual use -- see README)
!kaggle --version

## Quick smoke test

Create a small dataset, train an XGBoost model, and plot the results to verify the full stack is working end-to-end.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# Load sample data
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# Train XGBoost model
model = xgb.XGBClassifier(n_estimators=50, random_state=42, eval_metric="logloss")
model.fit(X_train, y_train)

# Evaluate
acc = accuracy_score(y_test, model.predict(X_test))
print(f"Test accuracy: {acc:.4f}")

# Plot feature importances
fig, ax = plt.subplots(figsize=(8, 5))
feat_imp = pd.Series(model.feature_importances_, index=data.feature_names).nlargest(10)
feat_imp.plot(kind="barh", ax=ax)
ax.set_title("Top 10 feature importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## GPU acceleration (NVIDIA only)

If you are using the **Kaggle NVIDIA** container, the cells below verify the GPU stack. On CPU or Mac containers they print a message and skip gracefully.

- **PyTorch** works on Pascal (P100 / GTX 10xx) and newer. This environment ships a custom torch build with Pascal support. **Note:**, as of 2026-07-02 torch is broken in Kaggle's own P100 notebook sessions.
- **RAPIDS (cuDF / cuML / CuPy)** requires Volta (compute capability 7.0) or newer. It does not work on Pascal, here or on Kaggle.

### Torch

In [ ]:
try:

    import torch

    if torch.cuda.is_available():
        x = torch.ones(1000, device="cuda") * 2
        print(f"torch {torch.__version__} -- {torch.cuda.get_device_name(0)}")
        print(f"GPU kernel op OK: {x.sum().item()}")

    else:
        print("No CUDA device detected -- expected on CPU and Mac containers.")

except Exception as e:
    print(f"torch GPU check failed: {e}")

### RAPIDS

In [ ]:
try:
    import cudf
    import cuml
    import cupy as cp

    print(f"cudf {cudf.__version__}, cuml {cuml.__version__}, cupy {cp.__version__}")
    cc = int(cp.cuda.Device(0).compute_capability)

    if cc >= 70:
        s = cudf.Series([1, 2, 3])
        print(f"cuDF GPU series sum: {s.sum()}")

    else:
        print(f"Compute capability {cc/10:.1f} < 7.0: RAPIDS kernels not supported on Pascal (same limitation on Kaggle).")

except ImportError:
    print("RAPIDS not available -- expected on CPU and Mac containers.")

except Exception as e:
    print(f"RAPIDS check failed: {e}")